
# 🧠 Redes Neuronales Recurrentes (RNN), LSTM y GRU

Este cuaderno presenta **conceptos teóricos** (en celdas Markdown) y **ejemplos prácticos en código** para:

- RNN simple
- LSTM
- GRU
- Generación de texto con LSTM (incluye *temperature sampling*)


En la sesión anterior abordamos la idea de las Redes Neuronales Recurrentes y el problema de memoria que podrían llegar a tener:
## 🔁 RNNs Simples: idea y ecuaciones

Una **RNN simple** procesa una secuencia paso a paso manteniendo un **estado oculto** \(memoria\) que se actualiza con cada nueva entrada.

Actualización del estado en el paso \(t\):
$$
h_t = \phi(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
$$

Salida correspondiente:
$$
y_t = W_{hy} h_t + b_y
$$

**Diferencia frente a redes *feed-forward***: En las FFNN la salida depende solo de la entrada actual; mientras que en las RNN **también depende del estado pasado** \(contexto de la secuencia\).



## 🧩 LSTM — Long Short-Term Memory

**La memoria a corto y largo plazo (LSTM, por sus siglas en inglés)** es una versión mejorada de la red neuronal recurrente (RNN). Las LSTM pueden capturar dependencias a largo plazo en datos secuenciales, lo que las hace ideales para tareas como la traducción automática, el reconocimiento de voz y la predicción de series temporales. A diferencia de las RNN tradicionales, que utilizan un único estado oculto que se propaga a lo largo del tiempo, las LSTM introducen una celda de memoria que almacena información durante periodos prolongados, lo que permite abordar el desafío del aprendizaje de dependencias a largo plazo.

Las **LSTM**, al ser RNN's tienen como entrada el estado oculto anterior $h_{t-1}$ el vector $x_t$, pero se adiciona la entrada del **(cell state) estado de celda** anterior ($c_{t-1}$). Como salida tendrá la actualización del estado oculto $h_t$ (como vimos en las RNN's), y también la actualización del estado de celda $c_{t}$.

Las **LSTM**  introducen el concepto de **puertas/compuertas** que controlan el flujo de información, así como el **estado de celda** el cual está relacionada con la memoria a largo plazo.

![Arquitectura básica de una LSTM](https://upload.wikimedia.org/wikipedia/commons/6/63/Long_Short-Term_Memory.svg)


**Puerta de olvido**: La puerta de olvido $f$ decide qué valores del estado de celda anterior deben descartarse y cuáles deben conservarse.

Dos entradas, $x_t$ (entrada en el momento específico) y $h_{t-1}$ (estado oculto anterior), se introducen en la puerta y se multiplican por la matriz de pesos, seguidas de la adición de sesgo. El resultado se somete a una función de activación sigmoidea, que genera una salida en el rango $[0,1]$. Si, para un estado de celda específico, la salida es $0$ o cercana a $0$, la información se olvida; para una salida de $1$ o cercana a $1$, la información se conserva para uso futuro.

$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f),
$$

donde  $[h_{t-1}, x_t]$ representa la concatenación de los valores de los vectores $h_{t-1}$ y $x_t$.

**Puerta de entrada**: La adición de información útil al estado de celda se realiza mediante la puerta de entrada.

Primero, la información se regula mediante la función sigmoidea y se filtran los valores a recordar, de forma similar a la puerta de olvido, utilizando las entradas $h_{t-1}$ y $x_t$. Luego, se crea un **vector de candidatos** mediante la función $\tanh$ que genera una salida en el intervalo $[-1,1]$. Finalmente, los valores del vector y los valores regulados se multiplican, término a término, para obtener la información útil.



$$
i_t = \sigma(W_i [h_{t-1}, x_t] + b_i), \qquad \tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)
$$

**Actualización del estado de celda**:
Multiplicamos (término a término) el estado de celda anterior $C_{t-1}$ por $f_t$ filtrando así la información que habíamos decidido ignorar. Luego, añadimos la salida de la puerta de entrada, la cual representa los nuevos valores candidatos.

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$

**Puerta de salida y estado oculto**:
La puerta de salida es responsable de decidir qué parte del estado actual de la celda debe enviarse como estado oculto (salida) para este paso de tiempo.

Primero, la puerta utiliza una función sigmoidea para determinar qué información del estado actual de la celda se emitirá. Esto se realiza utilizando el estado oculto anterior $h_{t-1}$ y la entrada actual $x_t$
$$
o_t = \sigma(W_o [h_{t-1}, x_t] + b_o).
$$
A continuación, el estado actual de la celda $C_t$ se somete a una activación $\tanh$ para escalar sus valores entre $−1$ y $1$. Finalmente, este estado de celda transformado se multiplica elemento a elemento por $o_t$ para producir el estado oculto $h_t$:
$$
h_t = o_t \odot \tanh(C_t).
$$
Este estado oculto $h_t$ se pasa luego al siguiente paso de tiempo y también puede utilizarse para generar la salida de la red.





## 🧠 ¿Por qué las LSTM tienen “memoria más larga” que las RNN simples?

El núcleo de la ventaja de las LSTM está en su **estado de celda** $C_t$, el cual actúa como una **vía directa y controlada** para transportar información a lo largo del tiempo.

---

### 🧩 En una RNN simple

En una RNN tradicional, el estado oculto se actualiza así:

$$
h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
$$

Durante el entrenamiento, los gradientes deben propagarse a través de **múltiples multiplicaciones por $ W_{hh} $** y por derivadas de la función $ \tanh $, lo que hace que los valores tiendan a:

- **Desaparecer (vanishing)** si los valores son menores que 1,  
- **Explotar (exploding)** si son mayores que 1.

Esto provoca que la red **olvide información antigua** o se vuelva **numéricamente inestable** cuando las secuencias son largas.

---

### ⚙️ En una LSTM

La LSTM introduce un **estado de celda** $ C_t $ con una **ruta aditiva** (no multiplicativa) para la información:

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$

donde:

- $ f_t $ (puerta de olvido) controla **cuánto del estado previo** se conserva,  
- $ i_t $ (puerta de entrada) decide **cuánta información nueva** se añade.

---

### 🧮 Flujo de gradiente más estable

El flujo del gradiente a través del tiempo para el estado de celda es aproximadamente:

$$
\frac{\partial C_t}{\partial C_{t-1}} = f_t
$$

Por tanto, si la red aprende valores de $ f_t $ cercanos a 1 (no saturados), el gradiente se mantiene casi constante:

$$
\frac{\partial C_T}{\partial C_t} = \prod_{k=t+1}^{T} f_k \approx 1
$$

De esta forma, **el gradiente no desaparece** al propagarse a lo largo del tiempo, permitiendo que la LSTM **mantenga memoria de largo plazo**.

---

### 🧠 Intuición en palabras

- En una **RNN simple**, cada paso mezcla y distorsiona completamente la información anterior mediante transformaciones no lineales lo cual implica que la memoria se degrade rápidamente.  
- En una **LSTM**, el estado de celda $ C_t $ ofrece una **"línea de transmisión" de información** controlada por puertas.  
- Esto permite **recordar eventos relevantes durante cientos de pasos de tiempo**, mientras se olvidan los irrelevantes.

---

### 💬 En resumen

| Mecanismo | RNN simple | LSTM |
|------------|-------------|------|
| Actualización del estado | Multiplicativa y no lineal | Aditiva controlada |
| Gradiente a lo largo del tiempo | Tiende a desaparecer/explotar | Se mantiene estable si $ f_t \approx 1 $ |
| Memoria de largo plazo | ❌ No | ✅ Sí |
| Cont

## 🔢 Ejemplo numérico: puerta de olvido $f_t$ y actualización de $C_t$

Supón:
- Estado oculto previo $h_{t-1} \in \mathbb{R}^2$:
  $$
  h_{t-1} = \begin{bmatrix} 0.3 \\ -0.1 \end{bmatrix}
  $$
- Entrada actual $x_t \in \mathbb{R}^3$:
  $$
  x_t = \begin{bmatrix} 1.0 \\ 0.5 \\ -0.5 \end{bmatrix}
  $$
- Concatenación $[h_{t-1}, x_t] \in \mathbb{R}^5$:
  $$
  [h_{t-1}, x_t] = \begin{bmatrix} 0.3 \\ -0.1 \\ 1.0 \\ 0.5 \\ -0.5 \end{bmatrix}
  $$
- Tamaño del estado de celda/oculto: 2 (para simplificar).
----
###  Cálculo de la puerta de olvido $f_t$

Definimos pesos y sesgo de la puerta de olvido:
$$
W_f =
\begin{bmatrix}
0.2 & -0.4 & 0.1 & 0.3 & 0.0\\
-0.1 & 0.5 & -0.2 & 0.0 & 0.4
\end{bmatrix},
\quad
b_f =
\begin{bmatrix}
0.05\\
-0.10
\end{bmatrix}
$$

Primero, el pre-activación:
$$
z_f = W_f \,[h_{t-1}, x_t] + b_f
$$

Para la **primera neurona** (fila 1 de $W_f$):
$$
\begin{aligned}
z_{f,1} &=
0.2(0.3) + (-0.4)(-0.1) + 0.1(1.0) + 0.3(0.5) + 0.0(-0.5) + 0.05 \\
&= 0.06 + 0.04 + 0.10 + 0.15 + 0 + 0.05 = 0.40
\end{aligned}
$$

Para la **segunda neurona** (fila 2 de $W_f$):
$$
\begin{aligned}
z_{f,2} &=
-0.1(0.3) + 0.5(-0.1) + (-0.2)(1.0) + 0(0.5) + 0.4(-0.5) - 0.10 \\
&= -0.03 - 0.05 - 0.20 + 0 - 0.20 - 0.10 = -0.58
\end{aligned}
$$

Aplicamos sigmoide elemento a elemento:
$$
f_t = \sigma(z_f) \approx
\begin{bmatrix}
\sigma(0.40)\\ \sigma(-0.58)
\end{bmatrix}
=
\begin{bmatrix}
0.5987\\ 0.3589
\end{bmatrix}
$$

**Interpretación**: la primera dimensión conserva ≈ 59.9% del contenido previo; la segunda, ≈ 35.9%.

---

### Puerta de entrada e información candidata (para completar el contexto)

Tomemos (a modo de ejemplo coherente):

$$
i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \approx
\begin{bmatrix}
0.5150\\ 0.5474
\end{bmatrix},
\qquad
\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C) \approx
\begin{bmatrix}
-0.1145\\ 0.1391
\end{bmatrix}
$$

(Valores calculados con matrices $W_i, b_i, W_C, b_C$ particulares; no son únicos, solo ilustrativos.)

---

### Actualización del estado de celda

Supón el estado de celda previo:
$$
C_{t-1} = \begin{bmatrix} 0.8 \\ -0.4 \end{bmatrix}
$$

Entonces:
$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
=
\begin{bmatrix}
0.5987\\ 0.3589
\end{bmatrix}
\odot
\begin{bmatrix}
0.8\\ -0.4
\end{bmatrix}
+
\begin{bmatrix}
0.5150\\ 0.5474
\end{bmatrix}
\odot
\begin{bmatrix}
-0.1145\\ 0.1391
\end{bmatrix}
\approx
\begin{bmatrix}
0.4200\\ -0.0674
\end{bmatrix}
$$

---

### Gradiente a través del tiempo (intuición local)

El factor directo de paso del gradiente por el “canal de celda” es:
$$
\frac{\partial C_t}{\partial C_{t-1}} = f_t
$$

Con los valores del ejemplo:
$$
\frac{\partial C_t}{\partial C_{t-1}} \approx
\begin{bmatrix}
0.5987\\ 0.3589
\end{bmatrix}
$$

Si $f_t$ se mantiene **cerca de 1 (sin saturar la sigmoide)** a lo largo de varios pasos, la cadena de productos
$$
\prod_k f_k
$$
no colapsa a 0, lo que ayuda a **evitar el gradiente desapareciente** y **preservar memoria**.

---

### ⚠️ Nota sobre saturación

- Si $z_f \gg 0$, entonces $\sigma(z_f) \to 1$ **pero** $\sigma'(z_f) \to 0$: la puerta está **saturada** y el gradiente a través de **esa puerta** se atenúa.
- En la práctica, el entrenamiento aprende $z_f$ de forma que $f_t$ sea **alto pero no extremo**, manteniendo tanto **memoria** (al multiplicar por $C_{t-1}$) como **gradientes útiles** (derivadas no nulas).



## ⚙️ GRU — Gated Recurrent Unit

En Machine Learning las redes neuronales recurrentes (RNN) son esenciales para tareas que involucran datos secuenciales, como texto, voz y análisis de series temporales. Si bien las RNN tradicionales tienen dificultades para capturar dependencias a largo plazo debido al problema de la desaparición del gradiente, se desarrollaron arquitecturas como las redes de memoria a corto y largo plazo (LSTM) para superar esta limitación.

Sin embargo, las LSTM tienen una estructura muy compleja y un coste computacional elevado. Para superar este problema, se introdujo la **Unidad Recurrente con Compuerta (GRU)**, que utiliza la arquitectura LSTM fusionando sus mecanismos de compuerta, ofreciendo una solución más eficiente para muchas tareas secuenciales sin sacrificar el rendimiento.

La idea principal de las GRU es utilizar mecanismos de compuerta para actualizar selectivamente el estado oculto en cada paso de tiempo, lo que les permite recordar información importante y descartar detalles irrelevantes. Las GRU buscan simplificar la arquitectura LSTM fusionando algunos de sus componentes y centrándose en solo dos compuertas principales: la compuerta de actualización y la compuerta de reinicio.

![Arquitectura básica de una LSTM](https://upload.wikimedia.org/wikipedia/commons/5/5f/Gated_Recurrent_Unit.svg)

Puerta de actualización:  
Esta puerta decide cuánta información del estado oculto anterior debe conservarse para el siguiente paso de tiempo.
$$
z_t = \sigma(W_z [h_{t-1}, x_t] + b_z)
$$

Puerta de reinicio:  
Esta puerta determina cuánto del estado oculto del pasado debe olvidarse.
$$
r_t = \sigma(W_r [h_{t-1}, x_t] + b_r)
$$

Candidato para el estado oculto:  
Este es el posible nuevo estado oculto calculado en base a la entrada actual y el estado oculto anterior.
$$
\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t] + b_h), \qquad
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

Actualización del estado oculto:  
El estado oculto final es un promedio ponderado del estado oculto anterior y el estado oculto candidato basado en la puerta de actualización

$$
h_{t-1} + z_t\, \tilde{h}_t
$$
### 🔍 LSTM vs GRU (resumen)

| Característica | LSTM | GRU |
|---|---|---|
| Puertas | 3 (entrada, olvido, salida) | 2 (actualización, reinicio) |
| Estado de celda $C_t$ | Sí | No |
| Coste computacional | Más alto | Más bajo |
| Secuencias muy largas | Excelente | Muy bueno |
| Casos típicos | Texto, traducción | Voz/sensores, series cortas |



## 🔢 Ejemplo numérico completo de una red tipo GRU

Usaremos dimensiones pequeñas para ver todos los números.

**Suposiciones (dimensiones):**
- $h_{t-1} \in \mathbb{R}^2$, $x_t \in \mathbb{R}^3$  $\Rightarrow$ $[h_{t-1}, x_t] \in \mathbb{R}^5$
- Pesos por puerta: $W_\bullet \in \mathbb{R}^{2 \times 5}$, sesgos $b_\bullet \in \mathbb{R}^2$

**Valores:**
$$
h_{t-1}=
\begin{bmatrix}
0.3\\ -0.1
\end{bmatrix},
\quad
x_t=
\begin{bmatrix}
1.0\\ 0.5\\ -0.5
\end{bmatrix},
\quad
[h_{t-1},x_t]=
\begin{bmatrix}
0.3\\ -0.1\\ 1.0\\ 0.5\\ -0.5
\end{bmatrix}
$$
----
### Puerta de **actualización** $z_t$

$$
z_t=\sigma\!\big(W_z [h_{t-1},x_t] + b_z\big)
$$

Con:
$$
W_z=
\begin{bmatrix}
0.15 & -0.25 & 0.20 & 0.10 & -0.05\\
0.40 & \phantom{-}0.10 & -0.30 & 0.00 & \phantom{-}0.20
\end{bmatrix},
\quad
b_z=
\begin{bmatrix}
0.05\\ -0.02
\end{bmatrix}
$$

Pre-activación (detallado por neurona):
- Neurona 1:
  $$
  0.15(0.3)+(-0.25)(-0.1)+0.20(1.0)+0.10(0.5)+(-0.05)(-0.5)+0.05=0.395
  $$
- Neurona 2:
  $$
  0.40(0.3)+0.10(-0.1)+(-0.30)(1.0)+0(0.5)+0.20(-0.5)-0.02=-0.31
  $$

Aplicando sigmoide:
$$
z_t=\sigma\!\big([0.395,\,-0.31]^\top\big)\approx
\begin{bmatrix}
0.5975\\ 0.4231
\end{bmatrix}
$$

----
### Puerta de **reinicio** $r_t$

$$
r_t=\sigma\!\big(W_r [h_{t-1},x_t] + b_r\big)
$$

Con:
$$
W_r=
\begin{bmatrix}
-0.20 & 0.35 & 0.10 & -0.15 & \phantom{-}0.05\\
\phantom{-}0.10 & -0.05 & 0.25 & \phantom{-}0.20 & -0.10
\end{bmatrix},
\quad
b_r=
\begin{bmatrix}
0.00\\ 0.10
\end{bmatrix}
$$

Pre-activación:
- Neurona 1:
  $$
  -0.20(0.3)+0.35(-0.1)+0.10(1.0)-0.15(0.5)+0.05(-0.5)+0= -0.095
  $$
- Neurona 2:
  $$
  0.10(0.3)-0.05(-0.1)+0.25(1.0)+0.20(0.5)-0.10(-0.5)+0.10= 0.535
  $$

Sigmoide:
$$
r_t=\sigma\!\big([-0.095,\,0.535]^\top\big)\approx
\begin{bmatrix}
0.4763\\ 0.6306
\end{bmatrix}
$$

----
### Estado **candidato** $\tilde{h}_t$

Primero modulamos el pasado con $r_t$:
$$
r_t \odot h_{t-1} \approx
\begin{bmatrix}
0.4763\\ 0.6306
\end{bmatrix}
\odot
\begin{bmatrix}
0.3\\ -0.1
\end{bmatrix}
=
\begin{bmatrix}
0.1429\\ -0.0631
\end{bmatrix}
$$

Concatenamos con $x_t$:
$$
\big[r_t \odot h_{t-1},\,x_t\big]=
\begin{bmatrix}
0.1429\\ -0.0631\\ 1.0\\ 0.5\\ -0.5
\end{bmatrix}
$$

Ahora:
$$
\tilde{h}_t=\tanh\!\big(W_h [r_t \odot h_{t-1},x_t] + b_h\big)
$$

Con:
$$
W_h=
\begin{bmatrix}
0.30 & -0.20 & 0.15 & 0.05 & 0.10\\
-0.25 & \phantom{-}0.10 & -0.05 & 0.20 & -0.15
\end{bmatrix},
\quad
b_h=
\begin{bmatrix}
0.00\\ 0.03
\end{bmatrix}
$$

Pre-activación:
- Neurona 1:
  $$
  0.30(0.1429) + (-0.20)(-0.0631) + 0.15(1.0) + 0.05(0.5) + 0.10(-0.5) + 0
  \approx 0.1805
  $$
- Neurona 2:
  $$
  -0.25(0.1429) + 0.10(-0.0631) - 0.05(1.0) + 0.20(0.5) - 0.15(-0.5) + 0.03
  \approx 0.1129
  $$

Aplicando $\tanh$:
$$
\tilde{h}_t\approx
\begin{bmatrix}
\tanh(0.1805)\\ \tanh(0.1129)
\end{bmatrix}
\approx
\begin{bmatrix}
0.1785\\ 0.1125
\end{bmatrix}
$$

----
### Nuevo estado $h_t$

Regla de mezcla controlada por $z_t$:
$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$


Con los valores:
$$
1-z_t \approx
\begin{bmatrix}
0.4025\\ 0.5769
\end{bmatrix}
$$

Por componentes:
- Dim 1:
  $$
  h_t^{(1)} \approx 0.4025(0.3) + 0.5975(0.1785) \approx 0.2274
  $$
- Dim 2:
  $$
  h_t^{(2)} \approx 0.5769(-0.1) + 0.4231(0.1125) \approx -0.0101
  $$

Por tanto:
$$
h_t \approx
\begin{bmatrix}
0.2274\\ -0.0101
\end{bmatrix}
$$

---

### 🧠 Nota sobre gradientes (intuición)

En una GRU, si **congelamos** las puertas $z_t, r_t$ (mirada local), el término
$$
\frac{\partial h_t}{\partial h_{t-1}} \approx (1 - z_t)
$$
actúa como una **conexión residual controlada**: cuando $z_t$ es pequeño, se preserva más del pasado ($1-z_t$ grande) y el gradiente fluye mejor.  
En el caso real, el gradiente también fluye a través de $z_t$ y del **camino del candidato** $\tilde{h}_t$ (que depende de $r_t$ y de $h_{t-1}$), lo que añade rutas adicionales. Esta **mezcla aditiva controlada** es una de las razones por las que las GRU manejan mejor la memoria que una RNN simple.


# ⚙️ Configuración e importaciones

In [ ]:
from numpy import sqrt
# importanto únicamente la función sqrt

In [ ]:
from numpy import *
# importa toooodo paquete pero no le pone alias

In [ ]:
import numpy as np
# importa toooodo paquete con pone alias

In [ ]:
import os, numpy as np, tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Embedding, Dropout
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.keras.utils.set_random_seed(42)

print("TensorFlow:", tf.__version__)



## 🧪 Ejemplo simple: RNN, LSTM y GRU sobre datos sintéticos

A modo de demo, entrenamos tres modelos para **regresión** sobre secuencias aleatorias.


In [ ]:
from sample_data.models_utils import (
        set_seeds, get_synthetic_regression_data, build_seq_model,
        train_test_split_seq, fit_models, summarize_models,
        regression_metrics, predict_models
        )

In [ ]:
set_seeds(42)
X, y = get_synthetic_regression_data(batch=1024, timesteps=20, features=8, y_noise=0.05)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split_seq(X, y, test_size=0.2)

In [ ]:
input_shape = X_train.shape[1:]

models = {
    "RNN":  build_seq_model(kind="rnn",  input_shape=input_shape, units=64, dropout=0.2, learning_rate=1e-3),
    "LSTM": build_seq_model(kind="lstm", input_shape=input_shape, units=64, dropout=0.2, learning_rate=1e-3),
    "GRU":  build_seq_model(kind="gru",  input_shape=input_shape, units=64, dropout=0.2, learning_rate=1e-3),
}

In [ ]:
summarize_models(models)

In [ ]:
histories = fit_models(models, X, y, epochs=3, verbose=0)

In [ ]:
y_preds = predict_models(models, X_test, verbose=0)

In [ ]:
rows = []
for name, y_hat in y_preds.items():
    m = regression_metrics(y_test, y_hat)
    m["Modelo"] = name
    rows.append(m)

In [ ]:
df_metrics = pd.DataFrame(rows).set_index("Modelo").sort_values("MSE")
print("\nResultados en TEST (ordenado por MSE):")
display(df_metrics)

## 📊 Resumen de resultados y métricas (TEST)

### Resultados (ordenados por MSE)
| Modelo | MSE | MAE | R² |
|:--|--:|--:|--:|
| **LSTM** | 0.015276 | 0.090547 | **0.984962** |
| **GRU**  | 0.015625 | 0.091435 | 0.984618 |
| **RNN simple** | 0.094897 | 0.227522 | 0.906582 |

**Conclusión rápida:** LSTM y GRU rinden muy parecido y superan claramente a la RNN simple. LSTM es levemente mejor (≈2–3% por MSE/MAE) y ambas alcanzan R² ≈ 0.985, lo que indica que explican ~98.5% de la variabilidad del conjunto de prueba.

---

### Métricas: definición breve e interpretación

**MSE — Error cuadrático medio**
$$
\text{MSE} \;=\; \frac{1}{n}\sum_{i=1}^{n}\big(y_i - \hat{y}_i\big)^2
$$
- **Qué mide:** promedio del cuadrado de los errores.  
- **Interpretación:** penaliza más los errores grandes; útil para optimización y diagnóstico fino (sensibilidad a outliers).  
- **Mejor si:** **más bajo**.

**MAE — Error absoluto medio**
$$
\text{MAE} \;=\; \frac{1}{n}\sum_{i=1}^{n}\big|y_i - \hat{y}_i\big|
$$
- **Qué mide:** magnitud promedio del error en las mismas unidades del objetivo.  
- **Interpretación:** más **robusto** a outliers que el MSE; fácil de explicar a públicos no técnicos.  
- **Mejor si:** **más bajo**.

**R² — Coeficiente de determinación**
$$
R^2 \;=\; 1 \;-\; \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
$$
- **Qué mide:** proporción de la varianza de $y$ explicada por el modelo.  
- **Interpretación:** cercano a **1** implica excelente ajuste; puede ser negativo si el modelo es peor que predecir $\bar{y}$.  
- **Mejor si:** **más alto**.

---

### Lectura pedagógica de tus números
- **LSTM**: mejor equilibrio error/explicación (MSE y MAE más bajos; **R² ≈ 0.985**).  
- **GRU**: rendimiento casi indistinguible de LSTM con una ligera desventaja marginal.  
- **RNN simple**: errores sustancialmente mayores y **R² ≈ 0.907**, consistente con su menor capacidad para retener dependencias temporales largas (problema de gradiente desapareciente).


In [ ]:
best_model_name = df_metrics.index[0]
print(f"\nMejor modelo según MSE: {best_model_name}")
print("Primeras 5 predicciones vs. valor real:")
print(
    pd.DataFrame(
        {
            "y_true": y_test.reshape(-1)[:5],
            "y_pred": y_preds[best_model_name].reshape(-1)[:5],
            "error": (y_test.reshape(-1) - y_preds[best_model_name].reshape(-1))[:5],
        }
    )
)


## ✍️ Generación de texto con LSTM

Ejemplo mínimo de **modelado de lenguaje a nivel de caracteres**:
1. Construimos secuencias de longitud fija \(n\).
2. Entrenamos una LSTM para predecir el **siguiente carácter**.
3. Generamos texto **muestreando** de la distribución de salida con **temperatura** \\(T\\).

**Muestreo con temperatura**:
$$
p_i = \frac{\exp(\log(\hat{p}_i)T)}{\sum_j \exp(\log(\hat{p}_j)T)}
$$
- **T = 1**: muestreo neutro.  
- **T < 1**: más conservador.  
- **T > 1**: más creativo/aleatorio.


In [ ]:
from char_gen import (
    set_seeds,
    build_char_vocab,
    make_char_sequences,
    one_hot_encode,
    build_lstm_char_model,
    generate_text,
)


In [ ]:
text = (
    "el lenguaje natural es fascinante y poderoso. "
    "las redes recurrentes aprenden dependencias en secuencias. "
    "las lstm y las gru ayudan a mantener memoria a largo plazo."
)

set_seeds(42)

In [ ]:
chars, stoi, itos = build_char_vocab(text)
vocab_size = len(chars)
seq_len = 40

In [ ]:
X_idx, y_idx = make_char_sequences(text, seq_len, stoi)
X_oh, y_oh = one_hot_encode(X_idx, y_idx, vocab_size)

print(f"Vocab size: {vocab_size}")
print(f"X shape: {X_oh.shape} | y shape: {y_oh.shape}")

In [ ]:
model = build_lstm_char_model(seq_len=seq_len, vocab_size=vocab_size, units=128, learning_rate=1e-3)
model.summary()

history = model.fit(X_oh, y_oh, epochs=20, batch_size=64, verbose=0)
print("Modelo de texto entrenado ✅")

In [ ]:
# Generación con distintas temperaturas
seed = "las redes"
for T in [0.5, 1.0, 1.2]:
    print(f"\n--- Generación (T={T}) ---")
    print(generate_text(model, seed=seed, stoi=stoi, itos=itos,
                        seq_len=seq_len, vocab_size=vocab_size,
                        length=250, temperature=T))



## 🧾 Conclusiones

- **RNN simples**: útiles para dependencias cortas; sufren gradientes inestables en secuencias largas.  
- **LSTM**: añaden **estado de celda** y puertas; mejor estabilidad para largo plazo.  
- **GRU**: simplifican LSTM con rendimiento similar y menor coste.  
- **Generación de texto**: la **temperatura** controla creatividad vs. precisión.
